# 🚀 ACE-Net Baseline: Few-Shot Domain Adaptation & Held-Out (N=700) Evaluation
### Complete Seed 42 Protocol Matching DeepSentinel's Experimental Setup

### 🌟 5-Step Workflow:
1. **Step 1:** Connect GPU & Mount Google Drive
2. **Step 2:** Clone / Pull Repository & Install Dependencies
3. **Step 3:** Fast Selective Unzip (Extracts target MP4 videos for adaptation + test in ~20s)
4. **Step 4:** Run Few-Shot Adaptation Fine-Tuning (300 clips, 5 epochs, LR=5e-6) $\to$ `acenet_adapted.pth`
5. **Step 5:** Run Held-Out Benchmark Evaluation on 700 clips $\to$ DeLong Statistical Significance Test vs. DeepSentinel!


## Step 1: Connect to GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import torch

drive.mount('/content/drive')
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name  :', torch.cuda.get_device_name(0))


## Step 2: Clone or Pull Repository & Install Dependencies

In [ ]:
import os
%cd /content
if not os.path.exists('/content/Baseline_Training'):
    !git clone https://github.com/gjvlio/Baseline_Training.git
%cd /content/Baseline_Training
!git checkout feat/baseline-preprocessing-jc
!git pull
!pip install -q scikit-learn transformers librosa
print('✅ Environment ready!')


## Step 3: Fast Selective Unzip of Target Videos (~20 Seconds)
Selectively extracts only the clips needed for adaptation (300) and evaluation (700) directly from `fakeavceleb.zip`.

In [ ]:
%cd /content/Baseline_Training

!python -m scripts.extract_700_videos \
    --zip-path '/content/drive/MyDrive/THESIS_MOTHERFILE/datasets/fakeavceleb.zip' \
    --manifest 'Manifests/fakeavceleb_adapt_300.csv,Manifests/fakeavceleb_eval_700.csv' \
    --output-dir '/content/fakeav_raw'


## Step 4: Run Few-Shot Adaptation Fine-Tuning (Seed 42 Protocol, ~30s)
Fine-tunes the ACE-Net discriminator on the 300 adaptation clips (150 Real / 150 Fake from Speaker Set A) at LR=5e-6 for 5 epochs.

In [ ]:
%cd /content/Baseline_Training

!python -u -m scripts.adapt_acenet_fewshot \
    --manifest 'Manifests/fakeavceleb_adapt_300.csv' \
    --raw-dir '/content/fakeav_raw' \
    --init-ckpt '/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/checkpoints/best_baseline_model.pth' \
    --output-ckpt '/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/checkpoints/acenet_adapted.pth' \
    --epochs 5 \
    --lr 5e-6 \
    --batch-size 16 \
    --device cuda


## Step 5: Run Held-Out Benchmark Evaluation on 700 Clips (~5 mins)
Evaluates the adapted ACE-Net model (`acenet_adapted.pth`) on the 700 held-out test clips and computes paired DeLong statistical significance vs. DeepSentinel.

In [ ]:
%cd /content/Baseline_Training

!python -u -m scripts.evaluate_fakeavceleb_700 \
    --manifest 'Manifests/fakeavceleb_eval_700.csv' \
    --raw-dir '/content/fakeav_raw' \
    --ckpt '/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/checkpoints/acenet_adapted.pth' \
    --output-csv '/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training/checkpoints/preds_acenet_adapted_700.csv' \
    --device cuda
